# RAG - Retrieval, Scalability from Scratch

- Exact search vs Approximate search
- Graph
- FAISS

Pipeline:

1. Documents
2. Chunk documents
3. Embedding
4. FAISS indexing
5. Query
6. Embedding (query)
7. ANN Search in FAISS index
8. Top-k chunks
9. LLM context
10. Response

## Facebook AI Similarity Search (FAISS)

A library including **indexing** methods:

- Flat Index
- IVF
- PQ
- OPQ
- HNSW
- IVF+PQ
- GPU search
- Binary indexes

### Flat Index

Index every vector, no approximation!

> Only use on small number of vectors (1,000)

But when we have billions, forget it.

In [2]:
from src import cosine_similarity

vectors = []
query = []

for vector in vectors:
    cosine_similarity(query, vector)

To overcome huge number of vectors, we can:

- Partition the space -> IVF
- Compress vectors -> PQ
- Use GPU
- Combine methods

### Inverted File Index (IVF)

- Run **clustering** > **store** vectors.
- k-means > centroids
- Search nearest centroid > inspect nearby vectors.

> But the nearest neighbour might not in the searched cluster.

Then FAISS allows search multiple clusters.

### Product Quantisation (PQ)

- Store tiny codes instead of values (e.g., store 64 bytes, not 1536 numbers).
- Lose information but fast.

### GPU

- Simply use GPU for computations.

### Combine Methods

- Google-scale systems:

> Query > Embedding > IVF > HNSW inside cluster > PQ compression > Top 100 candidates > Exact cosine similarity > Top 10 documents.

- Approximation first, exact later.

## Build from Scratch

1. Build a simple IVF index from scratch

- Implement k-means.
- Assign vectors to clusters.
- Search only selected clusters.
- Measure the speed/accuracy trade-off against linear search.

2. Build a simple Product Quantisation prototype

- Split vectors into subvectors.
- Learn small codebooks.
- Compress vectors.
- Compare memory usage and retrieval quality.

3. Only then use FAISS

- Understand `IndexFlatL2`, `IndexFlatIP`, `IndexIVFFlat`, `IndexHNSWFlat`, `IndexIVFPQ`, and why each exists.
- You'll recognise them as implementations of concepts you've already built rather than mysterious APIs.

### IVF Index From Scratch

- Clustering
- Centroids
- k-means

#### K-means

1. Random centroids
2. Assign points to clusters
3. Update centroids with mean of dimensions

Formula:

```math
\sum_{i=1}^K \sum_{x \in C_i}||x - \mu_i||^2
```

- $C_i$ := set of points in cluster i
- $\mu_i$ := centroid (mean) of that cluster

First, build KMeans class for the formula.

- `centroids`: stores coordinates of centroids, `shape = (k, embedding_dimension)`.
- `labels`: `np.array` type, tells which cluster every vector belongs to, parallel to the vector list.

Think of algorithm, we have:

Random centroids > Assign every points > Move centroids > Repeat.

The class also have these methods:

- init centroids
- assign clusters
- update centroids
- fit

In [ ]:
class KMeans:

    def __init__(self, k: int):
        self.k = k
        self.centroids = None
        self.labels = None

    def initialise_centroids():
        ...
    
    def assign_clusters(self, vectors: np.ndarray):
        '''
        Assign input vector to a centroid.
        '''
        if self.centroids is None:
            return "No centroids created."
        
        self.labels = np.empty(len(vectors), dtype=int)
        
        for i in range(len(vectors)):
            max_score = -np.inf
            best_centroid = None

            for j, centroid in enumerate(self.centroids):
                score = cosine_similarity(vectors[i], centroid)
                if score > max_score:
                    max_score = score
                    best_centroid = j
            
            self.labels[i] = best_centroid

    def update_centroids(self):
        '''
        Average points in a cluster to get the actual centroid.
        '''
        ...

    def fit(self, vectors):
        '''
        Once centroids are moved, they change which vectors belong to them.
        We have to do the process again and again till Convergence.
        '''
        ...

#### Inverted File

Instead of a big list of vectors, we keep an inverted mapping from each centroid to the vectors assigned to it.

- Centroid 0 -> list of vector IDs
- Centroid 1 -> list of vector IDs
- ...


#### Full Flow

Query > Nearest Centroid > Retrieve assigned vectors > Cosine Similarity > Top K.